In [1]:
USER_TIME_TO_SPEAK = 10

DEFAULT_TOPIC = '''
这是一个喜剧剧本讨论室，您可以自由地交谈。
'''

SYS_PROMPT = '''
您可以指定一个成员回复您的消息，可以使用@符号。
这意味着在您的消息中包含@符号，然后是
该人的名字，并在名字后面留一个空格。
所有参与者都是：{agent_names}
'''

In [2]:
import agentscope

modelscope_cfg_dict = "dashscope/model_config.json"
modelscope_agent_dict = "dashscope/agent_config.json"

npc_agents = agentscope.init(
    model_configs=modelscope_cfg_dict,
    agent_configs=modelscope_agent_dict,
    project="Conversation with Mentions",
    save_api_invoke=True,
)


2024-09-21 23:02:07.763 | INFO     | agentscope.manager._model:load_model_configs:115 - Load configs for model wrapper: my_dashscope_chat_config
2024-09-21 23:02:07.778 | INFO     | agentscope.models.model:__init__:203 - Initialize model by configuration [my_dashscope_chat_config]
2024-09-21 23:02:07.779 | INFO     | agentscope.models.model:__init__:203 - Initialize model by configuration [my_dashscope_chat_config]
2024-09-21 23:02:07.779 | INFO     | agentscope.models.model:__init__:203 - Initialize model by configuration [my_dashscope_chat_config]


In [3]:
from agentscope.agents import UserAgent

user = UserAgent()

agents = list(npc_agents) + [user]
agents

In [4]:
from agentscope.message import Msg

hint = Msg(
    name="Host",
    content = DEFAULT_TOPIC + SYS_PROMPT.format(
        agent_names=[agent.name for agent in agents],
    ),
    role="assistant"
)

hint.content

"\n这是一个喜剧剧本讨论室，您可以自由地交谈。\n\n您可以指定一个成员回复您的消息，可以使用@符号。\n这意味着在您的消息中包含@符号，然后是\n该人的名字，并在名字后面留一个空格。\n所有参与者都是：['Guo', 'Zhao', 'Chen', 'User']\n"

In [5]:
import re
from typing import Sequence

def select_next_one(agents:Sequence, rnd: int)->Sequence:
    """
    Select next agent
    """
    return agents[rnd % len(agents)]

def filter_agents(string: str, agents: Sequence) -> Sequence:
    """
    该函数会筛选输入字符串中以'@'为前缀的给定名称的出现，并返回找到的名称列表。
    """

    # 如果没有传入任何代理，则直接返回空列表
    if len(agents) == 0:
        return []

    # 创建一个匹配@后跟任何候选名字的模式
    pattern = (
        r"@(" + "|".join(re.escape(agent.name) for agent in agents) + r")\b"
    )

    # 在字符串中找到所有模式的出现
    matches = re.findall(pattern, string)

    # 为了快速查找，创建一个将代理名映射到代理对象的字典
    agent_dict = {agent.name: agent for agent in agents}

    # 返回匹配的代理对象列表，保持顺序
    ordered_agents = [agent_dict[name] for name in matches if name in agent_dict]
    
    return ordered_agents


In [6]:
from agentscope.msghub import msghub

rnd = 0
speak_list = []
with msghub(participants=agents, announcement=hint):
    while True:
        x = user()
        if x.content == "exit":
            break
        
        speak_list += filter_agents(x.content, npc_agents)
        
        if len(speak_list) > 0:
            next_agent = speak_list.pop(0)
            x= next_agent()
        else:
            next_agent = select_next_one(npc_agents, rnd)
            x = next_agent

        # speak_list += filter_agents(x.content, npc_agents)

        rnd += 1

User: 
User: 现在开始@Guo
Guo: Guo: 好嘞！咱们来聊聊这剧本里的笑点。首先，得让这两位主角的性格差异鲜明起来。咱们的年轻艺人，咱们就叫他“艺”吧，他坚守传统艺术，一脸正气，而他的新朋友，咱们就叫他“村”，他是个来自农村的小品演员，幽默风趣，说话带点土味儿。这样，两个人一碰头，就有了火花！

艺: （正襟危坐，一本正经）村兄，你这小品里怎么总用些农村的梗啊？这不是老土了吗？

村: （哈哈大笑）哎呀艺兄，那可不，咱农村的事儿多着呢！再说，城里人不也爱听嘛，你说是不是？

艺: （无奈地摇摇头）可是，我总觉得这样会贬低了我们的传统艺术。

村: （拍了拍艺的肩膀）兄弟，别担心，咱可以用这些接地气的元素，给传统艺术加点现代感，让它活起来，吸引更多的观众。

艺: （点头）嗯，你说得有道理。那咱们怎么在对话里设计笑点呢？

村: （眼睛一亮）比如说，艺兄，你能不能在表演时突然模仿村姑的动作，比如搓手、瞪眼，然后突然又恢复正经，观众肯定乐坏了！

艺: （微笑）这个主意不错，但得把握好分寸，不能太过火，否则就变成闹剧了。

村: （嘿嘿一笑）放心，艺兄，我懂分寸。还有，咱们可以在对话中加入一些双关语或者方言梗，让笑声更自然。

艺: （思考片刻）比如，当我们在讨论如何吸引现代观众时，我可以提出“艺术需要与时俱进”，然后村兄你接一句：“那可不，咱得让艺术跟上时代的步伐，就像咱村里的小伙子们，以前只会种地，现在都成了网红主播呢！”

村: （鼓掌）妙哉！艺兄，你这台词设计得既贴合主题，又不失幽默，简直完美！

艺: （笑）咱们就这样定了，接下来咱们再想一想其他的情节和笑点，让这部喜剧既有深度，又有广度，既能展现传统艺术的魅力，又能贴近现代观众的口味。

村: （兴奋地）好嘞，艺兄，咱们这就开始吧！
User: Guo说的太长了，而且不够好笑@Zhao
Zhao: Zhao: 哈哈，说得对，我们得让剧本更加生动有趣。关于艺和村的角色设定，我觉得艺可以是一个有些固执的老派艺术家，他对传统艺术有着深厚的感情，总是坚信自己的方式是最好的。而村则是一个乐观、幽默且充满活力的人，他以自己独特的方式看待世界，总能发现生活中最简单却最让人开心的瞬间。

艺: （严肃地）村兄，你总是用那些农村的笑话，会不会显得我们传统艺术太土了？

村: （轻松地）艺兄，你误会了。那些农